In [1]:
import sqlite3
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import json
import warnings
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from IPython.display import display, HTML
import os

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


def load_data_universal(db_path):
    """
    Fixed Loader: 
    1. Uses correct column names (similarity_vector).
    2. Generates Z-scores automatically (for run_full_analysis).
    3. Computes Semantic Entropy (for Dashboard).
    """
    if not os.path.exists(db_path):
        print(f"❌ Database not found at {db_path}")
        return None, None, None

    conn = sqlite3.connect(db_path)
    
    # 1. Get Latest Experiment
    try:
        exp_df = pd.read_sql("SELECT experiment_id, name FROM Experiments ORDER BY experiment_id DESC LIMIT 1", conn)
        if exp_df.empty: return None, None, None
        latest_id, name = exp_df.iloc[0]['experiment_id'], exp_df.iloc[0]['name']
    except: return None, None, None

    print(f"Loading Run: {name} (ID: {latest_id})")
    
    # 2. Fetch Data (FIXED COLUMN NAMES)
    query = f"""
    SELECT 
        r.result_id, r.question_id_external, r.question_text, r.gold_answer, 
        r.predicted_answer, r.full_trace_text, r.is_correct,
        r.uq_avg_entropy as 'Entropy', 
        r.uq_min_logit_gap as 'LogitGap',
        r.uq_heuristic_score as 'Heuristic', 
        r.gpt_eval_reason as 'Reason',
        r.uq_mech_score as 'Mechanistic',
        r.uq_coherence as 'Consistency',   -- Use pre-computed Coherence (excludes self)
        r.similarity_vector,                   -- Correct column name
        r.uq_tokens, r.uq_mech_trace, r.uq_entropy_trace, r.uq_logit_gap_trace
    FROM Results r 
    WHERE r.experiment_id = {latest_id}
    """
    
    df = pd.read_sql(query, conn)
    conn.close()
    
    if df.empty: return None, None, None
    df['Correct'] = df['is_correct'].astype(int)
    
    # 3. Parse JSON Columns
    json_cols = ['uq_tokens', 'uq_mech_trace', 'uq_entropy_trace', 'uq_logit_gap_trace', 'similarity_vector']
    for col in json_cols:
        df[col] = df[col].apply(lambda x: json.loads(x) if x and isinstance(x, str) else [])

    # 4. Feature Engineering
    # Semantic Entropy (Group-wise)
    def calc_se(sub):
        answers = sub['predicted_answer'].fillna("").str.strip().str.lower()
        probs = answers.value_counts(normalize=True)
        return -np.sum(probs * np.log(probs + 1e-10))

    if 'question_id_external' in df.columns:
        try: se_map = df.groupby('question_id_external').apply(calc_se, include_groups=False)
        except TypeError: se_map = df.groupby('question_id_external').apply(calc_se)
        df['Semantic'] = df['question_id_external'].map(se_map)
    else:
        df['Semantic'] = 0.0

    df['Length'] = df['uq_tokens'].apply(len)
    df = df[df['Length'] <= 500] # Safety Filter

    # 5. Generate Z-Scores (CRITICAL FOR run_full_analysis)
    potential_metrics = ['Entropy', 'LogitGap', 'Heuristic', 'Mechanistic', 'Consistency', 'Semantic', 'Length']
    valid_metrics = []
    
    for m in potential_metrics:
        if m in df.columns and df[m].var() > 0:
            valid_metrics.append(m)
            # Create _Z column
            df[f'{m}_Z'] = (df[m] - df[m].mean()) / df[m].std()

    return df, valid_metrics, (latest_id, name)

In [2]:
import analysis_utils as utils
import pandas as pd
import numpy as np
import importlib

importlib.reload(utils)
print("Loading Data & Generating Artifacts...")
df, metrics, exp_info = load_data_universal("db/results.sqlite")

Loading Data & Generating Artifacts...
Loading Run: HUV_Benchmark_Optimized (ID: 1)


In [3]:
len(df[df["gold_answer"] != df["predicted_answer"]])

df.keys()

Index(['result_id', 'question_id_external', 'question_text', 'gold_answer',
       'predicted_answer', 'full_trace_text', 'is_correct', 'Entropy',
       'LogitGap', 'Heuristic', 'Reason', 'Mechanistic', 'Consistency',
       'similarity_vector', 'uq_tokens', 'uq_mech_trace', 'uq_entropy_trace',
       'uq_logit_gap_trace', 'Correct', 'Semantic', 'Length', 'Entropy_Z',
       'LogitGap_Z', 'Heuristic_Z', 'Mechanistic_Z', 'Consistency_Z',
       'Semantic_Z', 'Length_Z'],
      dtype='object')

In [4]:
# edf = df[(df["gold_answer"] == df["predicted_answer"]) & df["is_correct"]]

# display(Markdown(f"### Matched and Correct Rows: **{len(edf)}**"))

# for i, row in edf.iterrows():
#     display(Markdown(f"""
# ---

# **Predicted Answer**
# ```
# {row["predicted_answer"]}
# ```

# **Full Trace**
# ```
# {row["full_trace_text"]}
# ```

# **Gold Answer**
# ```
# {row["gold_answer"]}
# ```
# """))

In [5]:
from IPython.display import display, Markdown

edf = df[(df["gold_answer"] != df["predicted_answer"]) & df["is_correct"]]

display(Markdown(f"### Mismatched but Correct Rows: **{len(edf)}**"))

for i, row in edf.iterrows():
    display(Markdown(f"""
---

**Predicted Answer**
```
{row["predicted_answer"]}
```

**Full Trace**
```
{row["full_trace_text"]}
```

**Gold Answer**
```
{row["gold_answer"]}
```
"""))
    
    display(Markdown(row["Reason"]))

    display(Markdown(row["question_text"]))

    display(Markdown(str(row["result_id"])))

### Mismatched but Correct Rows: **665**


---

**Predicted Answer**
```
72
The answer is: 72
```

**Full Trace**
```
 In April, Natalia sold 48 clips.
In May, she sold half as many clips as in April, which is 48/2 = 24 clips.
Altogether, Natalia sold 48 + 24 = 72 clips in April and May.
#### 72
The answer is: 72
```

**Gold Answer**
```
72
```


Extracted: 72

Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

3

...

3988


---

**Predicted Answer**
```
3
```

**Full Trace**
```
 To determine how many fish Each person is given to eat, we need to follow these steps:

1. **Determine the total number of fish caught by Ittymangnark:**
   - Let \( F \) represent the number of fish caught.
   - They split the fish equally among themselves and Oomyapeck.
   - Therefore, each person gets \( \frac{F}{3} \) fish.

2. **Calculate the number of eyes per fish:**
   - Assume that each fish has \( E \) eyes (this value can vary but it's not necessary to find it explicitly).

3. **Calculate the number of eyes eaten by Oomyapeck:**
   - Oomyapeck eats \( 22 \) eyes from the fish he receives.

4. **Determine the number of eyes left for him to give to his dog:**
   - Since Oomyapeck eats \( 22 \) eyes, there are no eyes left for him to give to his dog because:
     \[
     22 \text{ eyes} = 22 \text{ eyes}
     \]
   - This implies that Oomyapeck does not receive any fish or eyes.

5. **Calculate the number of fish each person is given to eat:**
   - Since Oomyapeck doesn't get any fish, each person still gets \( \frac{F}{3} \) fish.
   - Given that there are three people sharing the fish, the amount of fish each person gets remains consistent regardless of whether Oomyapeck eats any fish.

Thus, each person is given:
\[
\boxed{\frac{F}{3}}
\]
```

**Gold Answer**
```
4
```


The student's calculation for determining the number of fish each person receives to eat is mathematically correct.

Ittymangnark and Kingnook are an Eskimo couple living in the most northern region of the Alaskan wilderness.  Together, they live with their child, Oomyapeck.  Every day Ittymangnark catches enough fish for the three of them to eat for the day and they split the fish equally between the three of them.  But after they have split the fish, they give Oomyapeck all of the eyes, who gives two of the eyes to his dog and eats the rest himself.  How many fish will each of them be given to eat if Oomyapeck eats 22 eyes in a day?

3991

In [6]:
# from IPython.display import display, Markdown

# edf = df[(df["gold_answer"] != df["predicted_answer"]) & df["is_correct"]]

# display(Markdown(f"### Mismatched but Correct Rows: **{len(edf)}**"))

# out = ""

# for i, row in edf.iterrows():
#     out += f"""
# ---

# **Predicted Answer**
# ```
# {row["predicted_answer"]}
# ```

# **Full Trace**
# ```
# {row["full_trace_text"]}
# ```

# **Gold Answer**
# ```
# {row["gold_answer"]}
# ```
# """

# print(out)